# LLMOps: The OpenAI API & Prompt Engineering

In [ ]:
# Day 45 — LLMOps: OpenAI & Prompt Engineering
#
# Machine Learning models used to predict numbers (regression) or categories (classification).
# Large Language Models (LLMs) predict TEXT.
#
# We are moving from MLOps to LLMOps.
# Instead of training a model for weeks, you call an API (like OpenAI's GPT-4)
# and give it instructions in English. This is called Prompt Engineering.
#
# Today we'll cover:
#   1. Setting up the OpenAI Client
#   2. System vs User Prompts
#   3. Zero-shot vs Few-shot prompting
#   4. Structured Output (JSON)
#
# Install: pip install openai python-dotenv

### 1. The OpenAI Client & Basic Chat

In [ ]:
# Note: To actually run this, you need an OPENAI_API_KEY environment variable.
# We will write the code structure so you understand how it works in a script.

openai_basic_code = '''import os
from openai import OpenAI

# The client automatically looks for os.environ.get("OPENAI_API_KEY")
client = OpenAI()

def ask_llm(question):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Choose the model (gpt-4o is smarter but more expensive)
        messages=[
            {"role": "user", "content": question}
        ],
        temperature=0.7 # 0 = strict/deterministic, 1 = creative/random
    )
    
    # Extract just the text from the complex response object
    return response.choices[0].message.content

if __name__ == "__main__":
    print(ask_llm("What is a Kubernetes Pod?"))
'''

print("Basic OpenAI Call Pattern:")
print(openai_basic_code)

### 2. System vs User Prompts

In [ ]:
# An LLM takes a list of "messages" as context.
# - 'system': The overarching rules and persona for the AI.
# - 'user': The actual request from the human.
# - 'assistant': The previous replies from the AI (used for chat memory).

system_prompt_code = '''
def devops_assistant(user_query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            # The SYSTEM prompt gives the AI its identity and constraints
            {
                "role": "system", 
                "content": "You are an elite Senior DevOps Engineer. You answer questions concisely. "
                           "You MUST provide bash commands wrapped in markdown code blocks. "
                           "If you don't know the answer, reply with 'MANUAL_REVIEW_NEEDED'."
            },
            # The USER prompt is what we want done right now
            {
                "role": "user", 
                "content": user_query
            }
        ]
    )
    return response.choices[0].message.content
'''

print("System Prompts direct the AI's behavior:")
print(system_prompt_code)

### 3. Zero-shot vs Few-shot Prompting

In [ ]:
# Zero-shot: Asking the AI to do a task with NO examples.
# Few-shot: Giving the AI a few examples so it perfectly mimics the output format.

few_shot_code = '''
def extract_server_names(log_text):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Extract the server name from the log."}, 
            # Example 1 (Shot 1)
            {"role": "user", "content": "Error on web-node-01: CPU at 99%"},
            {"role": "assistant", "content": "web-node-01"},
            # Example 2 (Shot 2)
            {"role": "user", "content": "db-backend-02 failed to connect"},
            {"role": "assistant", "content": "db-backend-02"},
            # The Actual Request
            {"role": "user", "content": log_text}
        ]
    )
    return response.choices[0].message.content
'''

print("Few-shot prompting is incredibly powerful for parsing unstructured data!")
print(few_shot_code)

### 4. Structured Output (JSON Mode)

In [ ]:
# If your Python script uses the LLM to make a decision, you don't want a paragraph of text.
# You want a JSON dictionary you can parse like `data['action']`.
# OpenAI supports 'json_object' mode.

json_mode_code = '''import json

def analyze_alert(alert_text):
    response = client.chat.completions.create(
        model="gpt-4-turbo", # GPT-4 is much better at strict JSON
        response_format={ "type": "json_object" }, # FORCE JSON OUTPUT
        messages=[
            {
                "role": "system", 
                "content": "You are a log analyzer. Output JSON with keys: 'severity' (High/Med/Low), "
                           "'component' (DB/API/UI), and 'action_required' (bool)."
            },
            {"role": "user", "content": alert_text}
        ]
    )
    
    json_string = response.choices[0].message.content
    # Convert the string into a Python dictionary
    return json.loads(json_string)

# If we ran this with: analyze_alert("Database connection timed out after 30s")
# It would return a dictionary: 
# {'severity': 'High', 'component': 'DB', 'action_required': True}
'''

print("JSON Mode for programmatic pipelines:")
print(json_mode_code)

In [ ]:
# Practice Questions:
#
# 1. What happens if you set temperature=0 for an LLM analyzing logs? Is this good or bad?
# 2. Why is 'json_object' mode essential when integrating LLMs into existing MLOps pipelines?
# 3. Write a System prompt for an LLM that reads Git commit messages and generates a release notes summary.

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The Automated Incident Responder
# You receive a PagerDuty alert containing a 500-line Java stack trace.
# Traditional DevOps involves reading this manually at 3 AM.
# 
# Write a Python script using the OpenAI API that:
# 1. Takes the stack trace as input.
# 2. Uses a System Prompt to act as a Java SRE.
# 3. Outputs JSON containing the exact file/line number causing the crash and a 1-sentence summary of the fix.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. LLMs are incredible at unstructured text parsing (like messy server logs). What used to take complex regex now takes a 1-line prompt.
# 2. AIOps 2.0 isn't just about detecting anomalies, it's about the AI explaining *why* it broke in plain English.